# Grocery exploratory data analysis

Author: Maria Jorda

Date: 22 Feb 2026

In this notebook 5 different datasets are loaded and merged into one, and an exploratory data analysis is also done. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Data download 

In [ ]:
# In order to pull the data you need to have the files in a folder in the same directory as this notebook. If you have the data in a different location, you can change the path in the code below.
relative_path="module_2_datasets/"

In [ ]:
df_orders = pd.read_parquet(f'{relative_path}orders.parquet')
print(df_orders.shape)
df_orders.head()

In [ ]:
df_items_asked_for = pd.read_parquet(f'{relative_path}regulars.parquet')
print(df_items_asked_for.shape)
df_items_asked_for.head()

In [ ]:
df_abandoned_cart = pd.read_parquet(f'{relative_path}abandoned_carts.parquet')
print(df_abandoned_cart.shape)
df_abandoned_cart.head()

In [ ]:
df_inventory = pd.read_parquet(f'{relative_path}inventory.parquet')
print(df_inventory.shape)
df_inventory.head()

In [ ]:
df_users = pd.read_parquet(f'{relative_path}users.parquet')
print(df_users.shape)
df_users.head()

## Sanity checks

### 1. Data types

In [ ]:
df_orders.dtypes

In [ ]:
df_items_asked_for.dtypes

In [ ]:
df_abandoned_cart.dtypes

In [ ]:
df_inventory.dtypes

In [ ]:
df_users.dtypes

In the first 4 datasets, data types are OK, but in df_users, first_ordered_at and customer_cohort_month are object but they refer to time, so we convert those

In [ ]:
df_users['first_ordered_at'] = df_users['first_ordered_at'].astype('datetime64[ns]')
df_users['customer_cohort_month']= df_users['customer_cohort_month'].astype('datetime64[ns]')
df_users.dtypes

### 2. Missing values

In [ ]:
df_orders.isna().sum()

In [ ]:
df_items_asked_for.isna().sum()

In [ ]:
df_abandoned_cart.isna().sum()

In [ ]:
df_inventory.isna().sum()

In [ ]:
print(df_users.shape)
df_users.isna().sum()


We only have missing values in users dataframe, in the variables of the sociodemographic info and in the count of people/animals variables. For 'user_nuts1' I think it's logical to change the missing values to a new value 'Missing', which can be valuable later for a model. For the 'count_x' variables there are a lot of missings probably because users don't waste time filling these values. But in an order there must be someone behind purchasing the items, usually an adult, so I think that a conservative way to deal with these missings is to put a 1 in count_adults and in count_people, and a 0 in the others, assuming that at least there is an adult behind each user. I'll also add bolean columns to account that those variables were missing. A different approach could be to delete these columns, but we'd lose more than 300 non-nulls. A different approach could also be to fill the NaNs with the median value of each variable, but I consider that we don't have enough non-nulls observations for those statistics to be reliable and applicable to all the observations.

In [ ]:
df_users['user_nuts1'] = df_users['user_nuts1'].fillna("Missing")
df_users['count_people_missing'] = df_users['count_people'].isna()
df_users['count_adults_missing'] = df_users['count_people'].isna()
df_users['count_children_missing'] = df_users['count_people'].isna()
df_users['count_babies_missing'] = df_users['count_people'].isna()
df_users['count_pets_missing'] = df_users['count_people'].isna()
df_users['count_adults'] = df_users['count_adults'].fillna(1)
df_users['count_people'] = df_users['count_people'].fillna(1)
df_users['count_children'] = df_users['count_children'].fillna(0)
df_users['count_babies'] = df_users['count_babies'].fillna(0)
df_users['count_pets'] = df_users['count_pets'].fillna(0)


## Merging the datasets

Before combining all the datasets, we should rename the column 'created_at' in df_orders, df_items_asked_for and df_abandoned_cart.We also have to unpack 'ordered_items' in df_orders and 'variant_id' in df_abandoned_cart because we cannot join those datasets with df_inventory using 'variant_id' because it is a single value. We also have to rename the item id column in each dataset to do the joins on that column later. And we also rename the 'id' column in df_orders and df_abandoned_cart.

In [ ]:
df_orders = df_orders.rename(columns={'created_at': 'order_created_at'})
df_items_asked_for = df_items_asked_for.rename(columns={'created_at': 'asked_for_created_at'})
df_abandoned_cart = df_abandoned_cart.rename(columns={'created_at': 'abandoned_cart_created_at'})

In [ ]:
df_orders = df_orders.explode('ordered_items').rename(columns={'ordered_items': 'item_id'})
df_abandoned_cart = df_abandoned_cart.explode('variant_id').rename(columns={'variant_id': 'item_id'})

In [ ]:
df_items_asked_for = df_items_asked_for.rename(columns={'variant_id': 'item_id'})
df_inventory = df_inventory.rename(columns={'variant_id': 'item_id'})

In [ ]:
df_orders = df_orders.rename(columns={'id': 'order_id'})
df_abandoned_cart = df_abandoned_cart.rename(columns={'id': 'abandoned_cart_id'})

We can do the merge now.

In [ ]:
# First, we merge the orders and items_asked_for datasets on the item and user columns
# We use an outer join to keep all orders, even those that don't have corresponding items asked for, 
# and those that don't have corresponding orders, but people ask for them 
df_merged = pd.merge(df_orders, df_items_asked_for, on=['item_id', 'user_id'], how='outer') 

# Then, we merge the resulting dataset with the abandoned_cart dataset on the order and user columns
df_merged = pd.merge(df_merged, df_abandoned_cart, on=['item_id', 'user_id'], how='outer')

# Now we merge the resulting dataset with the inventory dataset on the item column
df_merged = pd.merge(df_merged, df_inventory, on='item_id', how='left')

# Finally, we merge the resulting dataset with the users dataset on the user column
df_merged = pd.merge(df_merged, df_users, on='user_id', how='left')

df_merged.head()


In [ ]:
df_merged.dtypes

Now that we have all the data together, we can do some analysis to understand the data deeper.

## Analysis

### 1. Converted items (from searching and saving or from abandoning the items to buying them)

We want to know if the items users ask for, are later purchased or not. Moreover, we want to analyze if abandoned items are later on bought.  

In [ ]:
# Conversion rate: from the items that a user asked for, how many were actually ordered?
df_asked_for = df_merged[df_merged['asked_for_created_at'].notna()]

# We filter the dataset to keep only one row per user-item pair, to avoid counting multiple times the same user asking for the same item
df_unique_asked_for = df_asked_for.drop_duplicates(subset=['user_id', 'item_id'])

number_of_inquiries = len(df_unique_asked_for)

number_of_purchases_asked = df_unique_asked_for[
        (df_unique_asked_for['order_created_at'].notna()) &
        (df_unique_asked_for['order_created_at'] >= df_unique_asked_for['asked_for_created_at'])
    ].shape[0]

conversion_rate_of_asked_products = number_of_purchases_asked / number_of_inquiries

print(f'From all times that a user asked for an item, a product was bought {conversion_rate_of_asked_products:.2%} of the times.')

In [ ]:
# Are abandoned items bought later on?
df_abandoned = df_merged[df_merged['abandoned_cart_created_at'].notna()]

# We filter the dataset to keep only one row per user-item pair, to avoid counting multiple times the same user abandoning the same item
df_unique_abandoned = df_abandoned.drop_duplicates(subset=['user_id', 'item_id'])

number_of_abandoned = len(df_unique_abandoned)

number_of_purchases_abandoned = df_unique_abandoned[
        (df_unique_abandoned['order_created_at'].notna()) &
        (df_unique_abandoned['order_created_at'] >= df_unique_abandoned['abandoned_cart_created_at'])
    ].shape[0]          

conversion_rate_of_abandoned_products = number_of_purchases_abandoned / number_of_abandoned

print(f'From all times that a user abandoned an item on the cart, a product was bought {conversion_rate_of_abandoned_products:.2%} of the times.')

In [ ]:
# Are abandoned products more likely to be bought than products that were asked for?
if conversion_rate_of_abandoned_products > conversion_rate_of_asked_products:
    print('Abandoned products are more likely to be bought than products that were asked for.')
elif conversion_rate_of_abandoned_products < conversion_rate_of_asked_products:
    print('Abandoned products are less likely to be bought than products that were asked for.')
else:
    print('Abandoned products and products that were asked for have the same likelihood of being bought.')

### 2. Product analysis

Let's now deepdive on the more common products that are bought

In [ ]:
df_purchases = df_merged[df_merged['order_created_at'].notna()]

In [ ]:
# Plot of most bought items

plt.figure(figsize=(10, 6))
top_10_most_bought_items = df_purchases['product_type'].value_counts().head(10)
sns.barplot(x=top_10_most_bought_items.index, y=top_10_most_bought_items.values, palette='viridis')
plt.title('Top 10 most bought product types')
plt.xlabel('Product type')
plt.ylabel('Number of purchases')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of prices of bought items
plt.figure(figsize=(10, 6))
sns.histplot(df_purchases['price'], bins=30, color='lightblue')
plt.title('Distribution of prices of bought items')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Impact of discount on the likelihood of a product being bought
df_purchases['has_discount'] = df_purchases['compare_at_price'] > df_purchases['price']
plt.figure(figsize=(10, 6))
sns.countplot(x='has_discount', data=df_purchases, palette='viridis')
plt.title('Impact of discount on the likelihood of a product being bought')
plt.xlabel('Has discount')
plt.ylabel('Number of purchases')
plt.tight_layout()
plt.show()


In [ ]:
# Most common vendors
plt.figure(figsize=(10, 6))
top_10_vendors = df_purchases['vendor'].value_counts().head(10)
sns.barplot(x=top_10_vendors.values, y=top_10_vendors.index, palette='viridis')
plt.title('Top 10 most common vendors')
plt.xlabel('Number of purchases')
plt.ylabel('Vendor')
plt.tight_layout()  
plt.show()

### 3. Users analysis

We now analyze the users information

In [ ]:
df_unique_users = df_merged.drop_duplicates(subset='user_id')

In [ ]:
# We combine 2 plots: one with the most common user locations among all users, and another one 
# with the most common user locations among users that bought products
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# More common user locations
top_10_locations = df_unique_users['user_nuts1'].value_counts().head(10)
sns.barplot(x=top_10_locations.values, y=top_10_locations.index, palette='viridis', ax=axes[0])
axes[0].set_title('Top 10 most common user locations (All users)')
axes[0].set_xlabel('Number of users')
axes[0].set_ylabel('Location')

# More common user locations among users that bought products
top_10_locations_bought = df_purchases['user_nuts1'].value_counts().head(10)
sns.barplot(x=top_10_locations_bought.values, y=top_10_locations_bought.index, palette='viridis', ax=axes[1])
axes[1].set_title('Top 10 most common user locations (Purchases)')
axes[1].set_xlabel('Number of purchases')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# More common user segments
plt.figure(figsize=(10, 6))
top_10_segments = df_unique_users['user_segment'].value_counts().head(10)
sns.barplot(x=top_10_segments.values, y=top_10_segments.index, palette='viridis')
plt.title('Most common user segments')   
plt.xlabel('Number of users')
plt.ylabel('User segment')
plt.tight_layout()
plt.show()

In [ ]:
# Do Top Up users buy more expensive products than other users?
plt.figure(figsize=(10, 6))
sns.boxplot(x='user_segment', y='price', data=df_purchases, palette='viridis')
plt.title('Price of bought products by user segment')
plt.xlabel('User segment')
plt.ylabel('Price')     
plt.tight_layout()
plt.show()

In [ ]:
# We now analyze the number of orders made by users
plt.figure(figsize=(10, 6))
fidelity = df_orders.groupby('user_id')['user_order_seq'].max()
sns.histplot(fidelity, bins=30, color='lightblue')
plt.title('Distribution of number of orders made by users')
plt.xlabel('Number of orders')
plt.ylabel('Number of users')
plt.tight_layout()
plt.show()

These graphs have given us a lot of information about our dataset (and we could keep digging into the data), which gives us a sense of the data and we could now start our modelling step to improve the conversion rate of abandoned items or asked items or for something else.